In [ ]:
import os
import copy
import subprocess
from glob import glob
from itertools import product
from datetime import datetime
from pathlib import Path
from string import Template
from utils.notebook import isnotebook
if isnotebook():
    home_dir = os.path.expanduser("~")
    os.chdir(os.path.join(home_dir, "aiwq"))

    # Autoreload modified packages
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

# Inline plotting setup
%matplotlib inline
%config InlineBackend.figure_formats = ['pdf', 'svg']
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Markdown, display
from AI_WQ_package import forecast_submission
from src.utils.data_io import *
from src.viz.viz_utils_pbc import *
from models.utils.general_util import printf
from models.utils.eval_util import get_target_dates
from models.utils.data_utils import get_measurement_variable
from models.utils.models_util import get_submodel_name, get_selected_submodel_name
from utils.timing import tic, toc
from utils.data_io import save_to_netcdf, load_data
from utils.logging import printf

# 
# Full set of regions, times, and tasks to evaluate
#
metrics = ["wtd_mse"]
gt_ids = [
    "era5-f1_tas", 
    "era5-f2_tas", 
    "era5-f3_tas", 
    "era5-f4_tas", 
    "era5-f1_pr", 
    "era5-f2_pr", 
    "era5-f3_pr", 
    "era5-f4_pr", 
    "era5-f1_mslp",
    "era5-f2_mslp",
    "era5-f3_mslp",
    "era5-f4_mslp",
]
# RPS are stored under the deterministic gt_ids
det_gt_ids = [
    "era5-tas", 
    "era5-pr", 
    "era5-mslp"
]
horizons = ["19", "26"]
target_eval_dates = ["std_test"]

# The full set of models we to evaluate in some experiment 
# De-biasing experiment model names
experiment_models = [
    "climatology", 
    "ecmwf",
    "proj_perpp_ecmwf", 
    "proj_tuned_ecmwfpp", 
    "pbc_ecmwf", 
    "debiased_ecmwf"
]


#### ECMWF, Debiased ECMWF, PBC-ECMWF barplots by task (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False

all_daily_rpss = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)



if False:
    print_improvements(all_daily_rpss, 
                       model_name='pbc_ecmwf_combo', 
                       baseline_models=['ecmwf', 'debiased_ecmwf'])

fig_show=True
fig_save=True
fig_by_season=False
plot_rpss_barplot(all_daily_rpss,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   by_season=fig_by_season)

#### ECMWF, Debiased ECMWF, PBC-ECMWF Diff maps (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_test"]
fig_verbose=False

all_daily_rpss = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)

# Set figure parameters
figure_model_names = ['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_metric = 'lat_lon_rpss'
figure_target_dates = 'std_test'
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
print_mean=False
figure_show = True
figure_save = True

plot_metric_diff_grid_6x4(model_names=figure_model_names,
                          temporal_rpss=all_daily_rpss,
                      gt_ids=figure_gt_ids,
                      horizons=figure_horizons,
                      metric=figure_metric,
                      target_dates=figure_target_dates,
                      diff_cmap=diff_cmap,
                      skill_cmap=skill_cmap,   
                      print_mean=print_mean,
                      show_fig=figure_show,
                      save_fig=figure_save)

#### AIFS vs. PBC-AIFS by task (2025)

In [ ]:
fig_model_names=['climatology', 'debiased_aifs', 'pbc_debias_aifs']
fig_model_names_str="AIFS models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_aifs_forecast"
fig_target_date_list=[fig_target_dates]
fig_verbose=False

all_daily_rpss = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_date_list,
                      verbose=fig_verbose)

if False:
    print_improvements(all_daily_rpss, 
                       model_name='pbc_debias_aifs', 
                       baseline_models=['debiased_aifs'])

fig_show=True
fig_save=True

plot_rpss_barplot(all_daily_rpss,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save) #, y_bottom=-0.2)

#### ECMWF, PoET, PBC-PoET barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'pbc_msn']
fig_model_names_str="MSN models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_msn_forecast"]
fig_verbose=False

all_daily_rpss = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)

if False:
    print_improvements(all_daily_rpss, 
                       model_name='pbc_msn', 
                       baseline_models=['msn', 'debiased_ecmwf'])

fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'pbc_msn']
fig_target_dates="std_msn_forecast"
fig_show=True
fig_save=True

plot_rpss_barplot(all_daily_rpss,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save)

#### Fuxi, PBC-ECMWF, barplots and diff maps (2017-2021)

In [ ]:
fig_model_names=['climatology', 'debiased_fuxi', 'pbc_ecmwf_combo']
fig_model_names_str="Fuxi models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_fuxi_ecmwf"]
fig_verbose=False

all_daily_rpss = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)

if True:
    print_improvements(all_daily_rpss, 
                       model_name='pbc_ecmwf_combo', 
                       baseline_models=['debiased_fuxi'])
    


fig_model_names=['climatology', 'debiased_fuxi', 'pbc_ecmwf_combo']
fig_target_dates="std_fuxi_ecmwf"
fig_show=True
fig_save=True

plot_rpss_barplot(all_daily_rpss,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save)

In [ ]:
fig_model_names=['debiased_fuxi', 'pbc_ecmwf_combo', 'climatology']
fig_model_names_str="FUXI models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_fuxi_ecmwf"]
fig_verbose=False

all_daily_rpss_fuxi = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)

# Set figure parameters
figure_model_names = ['debiased_fuxi', 'pbc_ecmwf_combo']
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_metric = 'lat_lon_rpss'
figure_target_dates = 'std_fuxi_ecmwf'
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
print_mean=False
figure_show = True
figure_save = True

plot_metric_diff_grid_6x3(model_names=figure_model_names,
                          all_daily_rpss=all_daily_rpss_fuxi,
                          gt_ids=figure_gt_ids,
                          horizons=figure_horizons,
                          metric=figure_metric,
                          target_dates=figure_target_dates,
                          diff_cmap=diff_cmap,
                          skill_cmap=skill_cmap, 
                          print_mean=print_mean,
                          show_fig=figure_show,
                          save_fig=figure_save)

#### ECMWF, Deb. ECMWF, PBC-ECMWF RPSS barplots by region (std_test: 2016-2024)

In [ ]:
fig_model_names=['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
fig_horizons = [19, 26]
fig_target_dates = ["std_test"]
fig_regions = 'all' 
fig_verbose=False

metrics_dic = get_daily_rpss(model_names = fig_model_names,
                               model_names_str="ECMWF-based models",
                               horizons = fig_horizons,
                               target_dates_list = fig_target_dates,
                               regions = fig_regions,
                               verbose=fig_verbose)

fig_model_names = ['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
fig_horizons = [19, 26]
fig_target_dates = "std_test"
fig_regions = 'all' 
fig_show = True
fig_save = True

plot_rpss_by_region_all(metrics_dic,
                       model_names=fig_model_names,
                       gt_ids=fig_gt_ids,
                       horizons=fig_horizons,
                       target_dates=fig_target_dates,
                       regions=fig_regions,
                       show_fig=fig_show,
                       save_fig=fig_save)



#### ECMWF, Debiased-ECMWF, PBC-ECMWF RPSS by season barplots (2016-2024)

In [ ]:
fig_model_names=["climatology", "ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
fig_model_names_str="ECMWF-based models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_test"]
fig_verbose=False

all_daily_rpss = get_daily_rpss(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)

# Set figure parameters
figure_model_names = ["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_target_dates = 'std_test'
figure_show = True
figure_save = True
figure_verbose = False

plot_seasonal_rpss_grouped_bar(all_daily_rpss,
                                   model_names=figure_model_names,
                                   gt_ids=figure_gt_ids,
                                   horizons=figure_horizons,
                                   target_dates=figure_target_dates,
                                   show_fig=figure_show,
                                   save_fig=figure_save,
                                   verbose=figure_verbose)

#### ECMWF, Debiased-ECMWF, PBC-ECMWF spatial bias (prediction-truth) (2016-2024)

In [ ]:
fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo", "gt"]
fig_gt_ids = ["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons = [19, 26]
fig_target_dates="std_test"
fig_vmin=-0.2
fig_vmax=0.2
fig_show_fig=True
fig_save_fig=True
fig_verbose=False

results_dict = get_all_preds(model_names=fig_model_names,
                  gt_ids=fig_gt_ids,
                  horizons=fig_horizons,
                  fs=[1, 2, 3, 4],
                  target_dates="std_test",
                  verbose=fig_verbose)

for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
    fig_show_cbar = (fig_horizon==26)
    plot_bias_maps_3x4(results_dict,  # Now taking the dictionary
                       model_names=fig_model_names,
                          gt_id=fig_gt_id,
                          horizon=fig_horizon,
                          fs=[1, 2, 3, 4],
                          cmap="RdBu_r",
                          vmin = fig_vmin,
                        vmax = fig_vmax,
                        show_cbar=fig_show_cbar,
                        show_fig = fig_show_fig,
                        save_fig = fig_save_fig
                    )

In [ ]:
if False:
    fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
    fig_gt_ids = ["era5-tas", "era5-pr", "era5-mslp"]
    fig_horizons = [19, 26]

    for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
        results = print_model_bias(results_dict, 
                        model_names=fig_model_names,
                        gt_id=fig_gt_id,
                        horizon=fig_horizon,
                        fs=[1, 2, 3, 4])